# 03 — Tracing & Observability

**Needs `GROQ_API_KEY`.** (Free — see the folder README.)

---

## A score tells you *that*. A trace tells you *why*.

Notebooks 01 and 02 gave you numbers: this answer scored 2/5. Useful — but the
number doesn't tell you **where it went wrong**. Did the agent retrieve the wrong
document? Call the wrong tool? Did one step take 8 seconds? Did a tool throw an
error the agent silently swallowed?

To answer those, you need **observability**: a record of *every step the agent
took*, with timings, inputs, outputs, token counts, and errors. The unit of that
record is a **span**, and a tree of spans for one run is a **trace**. This is the
exact model behind LangSmith, Langfuse, and OpenTelemetry.

We'll build a tiny tracer **from scratch** — maybe 40 lines — so the concept is
never magic. At the end we show how to get the same thing in production for free.

### What you'll build
1. A `Span` (one timed step) and a `Tracer` (collects spans into a tree)
2. A `@trace` **decorator** so any function records itself
3. An instrumented multi-step agent — watch the whole run light up
4. Real **token & cost capture** from Groq responses
5. A printed **trace tree** + aggregate observability metrics

## Step 1 — Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file (see the README)"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama3-8b-8192", temperature=0)
print("LLM ready:", llm.model_name)

## Step 2 — What is a span?

A **span** is a record of one unit of work. At minimum it captures:

| Field | Why it matters |
|-------|----------------|
| `name` | which step this was (`retrieve`, `llm_call`, ...) |
| `start` / `end` | so we can compute **duration** |
| `inputs` / `outputs` | what went in and came out — for debugging |
| `metadata` | extra facts: token counts, model, etc. |
| `error` | if the step blew up |
| `parent` | which span this nested inside — builds the **tree** |

We'll use a dataclass and a context manager so usage reads naturally:
`with tracer.span("retrieve"): ...`

In [ ]:
import time, uuid
from dataclasses import dataclass, field
from typing import Optional, Any

@dataclass
class Span:
    name: str
    span_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    parent_id: Optional[str] = None
    start: float = 0.0
    end: float = 0.0
    inputs: Any = None
    outputs: Any = None
    metadata: dict = field(default_factory=dict)
    error: Optional[str] = None

    @property
    def duration_ms(self) -> float:
        return round((self.end - self.start) * 1000, 2)

print("Span defined.")

## Step 3 — The Tracer

The `Tracer` holds all spans for a run and tracks the **current parent** so
nested `with` blocks automatically link up into a tree. The `span()` context
manager stamps the start/end times and records errors even when a step raises —
which is exactly when you most need the trace.

In [ ]:
from contextlib import contextmanager

class Tracer:
    def __init__(self):
        self.spans: list[Span] = []
        self._stack: list[str] = []   # ids of currently-open parent spans

    @contextmanager
    def span(self, name: str, inputs=None):
        s = Span(name=name, inputs=inputs,
                 parent_id=self._stack[-1] if self._stack else None)
        s.start = time.perf_counter()
        self.spans.append(s)
        self._stack.append(s.span_id)
        try:
            yield s                     # the `with` body runs here; set s.outputs inside
        except Exception as e:
            s.error = f"{type(e).__name__}: {e}"
            raise
        finally:
            s.end = time.perf_counter()
            self._stack.pop()

    def reset(self):
        self.spans.clear()
        self._stack.clear()

tracer = Tracer()

# Quick demo: a parent span with a child nested inside
with tracer.span("outer") as outer:
    time.sleep(0.01)
    with tracer.span("inner") as inner:
        time.sleep(0.02)
        inner.outputs = "done"

for s in tracer.spans:
    print(f"  {s.name:8} parent={s.parent_id}  {s.duration_ms} ms")

## Step 4 — A `@trace` decorator

Wrapping every function body in a `with` block is tedious. A **decorator** lets
any function trace itself automatically — its arguments become the span inputs
and its return value becomes the span outputs. This is how real tracing libraries
feel: add one line above a function and it shows up in your traces.

In [ ]:
import functools

def trace(name: str = None):
    def decorator(fn):
        span_name = name or fn.__name__
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            with tracer.span(span_name, inputs={"args": args, "kwargs": kwargs}) as s:
                result = fn(*args, **kwargs)
                s.outputs = result
                return result
        return wrapper
    return decorator

@trace()
def add(a, b):
    return a + b

tracer.reset()
print("result:", add(2, 3))
print("span captured:", tracer.spans[0].name, tracer.spans[0].outputs)

## Step 5 — Capture tokens & cost from a real LLM call

The most important thing to observe in an agent is the **LLM calls** — they're
the slow, expensive part. Groq (like every provider) returns token usage in the
response metadata. We wrap the call in a span and stash usage + estimated cost in
`metadata`, so every model call becomes auditable.

In [ ]:
# Made-up Llama-3-8b prices for the demo (per 1M tokens). Plug in real numbers for your provider.
PRICE_IN_PER_1M = 0.05
PRICE_OUT_PER_1M = 0.08

def traced_llm_call(prompt: str) -> str:
    with tracer.span("llm_call", inputs={"prompt": prompt}) as s:
        response = llm.invoke(prompt)
        usage = response.response_metadata.get("token_usage", {})
        in_tok = usage.get("prompt_tokens", 0)
        out_tok = usage.get("completion_tokens", 0)
        cost = in_tok / 1_000_000 * PRICE_IN_PER_1M + out_tok / 1_000_000 * PRICE_OUT_PER_1M
        s.metadata = {"model": llm.model_name, "prompt_tokens": in_tok,
                      "completion_tokens": out_tok, "cost_usd": round(cost, 8)}
        s.outputs = response.content
        return response.content

tracer.reset()
answer = traced_llm_call("In one sentence, what is observability in software?")
print("answer:", answer)
print("usage :", tracer.spans[0].metadata)

## Step 6 — Instrument a multi-step agent

Now the payoff. We build a small agent that (1) **plans** what to do, (2) calls a
**tool**, then (3) **writes** a final answer. Each step is a span, and the tool
call nests inside the agent span. When we print the trace, we'll *see the agent
think* — and spot exactly where time and tokens go.

In [ ]:
# A trivial 'tool' the agent can call
@trace("tool:word_count")
def word_count_tool(text: str) -> int:
    time.sleep(0.05)  # pretend this does real work
    return len(text.split())

@trace("agent")
def agent(question: str) -> str:
    # Step 1: plan (an LLM call)
    plan = traced_llm_call(f"You are a planner. In one short line, say how to answer: {question}")

    # Step 2: use a tool
    wc = word_count_tool(question)

    # Step 3: final answer (another LLM call)
    final = traced_llm_call(
        f"Question: {question}\nThe question has {wc} words.\nAnswer it in one sentence."
    )
    return final

tracer.reset()
result = agent("Why is the sky blue?")
print("\nFINAL ANSWER:", result)
print(f"\nCaptured {len(tracer.spans)} spans.")

## Step 7 — Print the trace tree

A flat list of spans is hard to read. Let's render them as an **indented tree**
(children under their parents), showing duration and any captured metadata. This
is the single most useful debugging view you can have for an agent.

In [ ]:
def print_trace(tracer: Tracer):
    by_parent = {}
    for s in tracer.spans:
        by_parent.setdefault(s.parent_id, []).append(s)

    def walk(parent_id, depth):
        for s in by_parent.get(parent_id, []):
            indent = "  " * depth
            tag = "❌ " + s.error if s.error else ""
            extra = ""
            if "prompt_tokens" in s.metadata:
                m = s.metadata
                extra = f"  [{m['prompt_tokens']}+{m['completion_tokens']} tok, ${m['cost_usd']:.6f}]"
            print(f"{indent}• {s.name:18} {s.duration_ms:>8.2f} ms{extra} {tag}")
            walk(s.span_id, depth + 1)

    print("TRACE")
    walk(None, 0)

print_trace(tracer)

### Reading the tree

You can now answer questions a score never could:
- **Where did the time go?** Compare the `ms` per span — usually one `llm_call`
  dominates.
- **How many tokens did the whole run burn?** Sum the `tok` tags.
- **Did a tool fail?** A red `❌` would jump out instantly.

This is *observability*: the run is no longer a black box.

## Step 8 — Aggregate observability metrics

From the raw spans we can compute the operational numbers teams watch on
dashboards: total latency, total tokens & cost, the **slowest** span (your
bottleneck), and the **error rate**.

In [ ]:
def observability_report(tracer: Tracer) -> dict:
    root = [s for s in tracer.spans if s.parent_id is None]
    total_ms = sum(s.duration_ms for s in root)  # root spans = wall-clock-ish total
    total_tokens = sum(s.metadata.get("prompt_tokens", 0) + s.metadata.get("completion_tokens", 0)
                       for s in tracer.spans)
    total_cost = sum(s.metadata.get("cost_usd", 0) for s in tracer.spans)
    errors = [s for s in tracer.spans if s.error]
    slowest = max(tracer.spans, key=lambda s: s.duration_ms)
    return {
        "spans": len(tracer.spans),
        "total_latency_ms": round(total_ms, 2),
        "total_tokens": total_tokens,
        "total_cost_usd": round(total_cost, 8),
        "slowest_span": f"{slowest.name} ({slowest.duration_ms} ms)",
        "error_rate": round(len(errors) / len(tracer.spans), 2),
    }

for k, v in observability_report(tracer).items():
    print(f"  {k:<18}: {v}")

## Step 9 — Errors are captured too

Observability earns its keep when things break. Because the tracer records errors
in the `finally` block, a crashing step still leaves a span behind — so you can
see *exactly* which step failed and what it received. Let's force a failure.

In [ ]:
@trace("flaky_tool")
def flaky_tool(x):
    raise ValueError("upstream API timed out")

tracer.reset()
try:
    with tracer.span("agent_run"):
        flaky_tool(42)
except ValueError:
    pass  # the agent would handle/retry; we just want the trace

print_trace(tracer)
print("\nThe failed span was still captured, with its error message — that's the point.")

## Step 10 — In production, you don't write this yourself

Building the tracer taught you the model. In real projects you flip on a hosted
tracer and get trace trees, dashboards, token/cost tracking, and search for free.

**The easiest one with LangChain:** add these to your `.env` and *every*
LangChain/LangGraph call is traced automatically — no code changes:

```
LANGCHAIN_API_KEY=lsv2_xxxxxxxxxxxx
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=Agent-Evaluation-And-Observability
```

Then open https://smith.langchain.com to see the exact tree you just built — but
interactive, searchable, and shared with your team.

| Tool | Good for |
|------|----------|
| **LangSmith** | Tightest LangChain/LangGraph integration |
| **Langfuse** | Open-source, self-hostable |
| **Arize Phoenix** | OSS, strong on RAG/embeddings analysis |
| **OpenTelemetry** | Vendor-neutral standard; export anywhere |

The cell below checks whether you have tracing switched on.

In [ ]:
if os.getenv("LANGCHAIN_TRACING_V2") == "true" and os.getenv("LANGCHAIN_API_KEY"):
    print("✅ LangSmith tracing is ON — run any cell above and check smith.langchain.com")
else:
    print("ℹ️  LangSmith tracing is OFF. Add the 3 env vars above to enable it (optional).")

## Step 11 — Try it yourself

1. **Add a step.** Give `agent` a second tool call and re-run Steps 6–8 — watch a
   new span appear and the token total climb.
2. **Find the bottleneck.** Add `time.sleep()` to one tool and see it become the
   `slowest_span`.
3. **Export the trace.** Write `tracer.spans` to a JSON file (hint: `dataclasses.asdict`)
   so you could load it into another tool.

In [ ]:
# Your playground — export the trace to JSON.
import dataclasses
tracer.reset()
agent("What is observability?")
trace_json = [dataclasses.asdict(s) for s in tracer.spans]
print(json.dumps(trace_json[0], indent=2, default=str)[:400])

## Summary

| What you built | What it taught |
|----------------|----------------|
| `Span` dataclass | One timed step: name, timing, IO, metadata, error |
| `Tracer` + `span()` | Nesting via a parent stack → a trace **tree** |
| `@trace` decorator | Instrument any function with one line |
| `traced_llm_call` | Capture real token usage & cost from the response |
| `print_trace` | The single best agent-debugging view |
| `observability_report` | Latency, tokens, cost, slowest span, error rate |

**The big takeaway:** evaluation scores the *output*; observability records the
*process*. You need both — a low score plus its trace tells you not just that the
agent failed, but exactly which step to fix.

**Next up → `04-rag-agent-metrics`:** real agents retrieve documents and call
tools. We'll measure whether retrieval fetched the right facts, whether the answer
stayed grounded in them, and whether the right tool was called.